Importation des bibliothèques

In [1]:
import os
import random
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import copy #pour copier base_model
import numpy as np
import pandas as pd
import itertools
import csv

In [2]:
from models import SimpleCNN, CNN_MCdropout
from dataset import load_cifar10
from train import train_model, evaluate
from utils import mc_predict_mean_probs, generate_mc_outputs
from accuracy import accuracy_threshold, monotonic_rearrangement, isotonic_regression, monotonicity_penalty

In [4]:
print(os.getcwd())  # donne le répertoire courant

c:\Users\Invite\Documents\INRIA\MCDropout


Fixation du seed pour la reproductibilité

In [4]:
def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

Configuration de base

Importation du modèle déjà entraîné par l'utilisateur

In [5]:
trainloader, valloader, testloader, classes = load_cifar10(batch_size=128, val_ratio=0.1)

In [3]:
device = torch.device("cuda")
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0)) #c'est censé être RTX

GPU Available: True
GPU Name: NVIDIA GeForce RTX 4060


In [ ]:
save_path = "best.pt"

# Vérifie si les poids existent déjà
base_model = SimpleCNN()
if os.path.exists(save_path):
    print("Chargement du modèle sauvegardé")
    base_model.load_state_dict(torch.load(save_path, map_location=device))  # même architecture que celle qui a sauvegardé
else:
    print("Pas de modèle sauvegardé, on entraîne le modèle")
    base_model = train_model(base_model, trainloader, valloader, device, epochs=20, save_path=save_path)
    base_model.load_state_dict(torch.load(save_path, map_location=device))  # recharge les meilleurs poids

Chargement du modèle sauvegardé


Test manuel de plusieurs dico_layers et stockage des résultats

In [ ]:
layer_names = ["conv1", "conv2", "conv3", "fc1"]
dropout_values = [round(x, 1) for x in np.arange(0.1, 1, 0.1)]

grid_configs = []
for r in range(1, len(layer_names)+1):
    for layers in itertools.combinations(layer_names, r):
        for p in dropout_values:
            dico = {layer: p for layer in layers}
            for before in [True, False]:
                grid_configs.append({
                    "dico_layers": dico,
                    "before": before
                })

print(f"Nombre total de configurations : {len(grid_configs)}")

Nombre total de configurations : 300
{'dico_layers': {'conv1': 0.1}, 'before': True}


Grid search sur dico_layers et stockage des résultats

In [ ]:
import pickle

all_results = [] 
all_accuracy_curves = []

metrics_list = [
    "variance_predicted",
    "variance_max",
    "predictive_entropy_predicted",
    "predictive_entropy_max",
    "relative_norm"
]

min_penalties = {metric: {"penalty_isotone": float("inf"), "penalty_rearrangement": float("inf"), "config_ids_iso": [], "config_ids_rearr": []} for metric in metrics_list}

for i, config in enumerate(grid_configs):
    dico_layers = config["dico_layers"]
    before = config["before"]

    model_mc = CNN_MCdropout(copy.deepcopy(base_model), dico_layers=dico_layers, before=before).to(device)
    
    all_probs = []
    all_Y = []
    all_metric_values = {metric: [] for metric in metrics_list}
    all_mean_probs = []
    
    for X, Y in testloader:
        X, Y = X.to(device), Y.to(device)
        probs, _ = mc_predict_mean_probs(model_mc, X, T=100, verbose=False)
        all_probs.append(probs)
        all_Y.append(Y)
        outputs, mean_probs, metric_values, _, _ = generate_mc_outputs(
            model_mc, X, T=100, metrics=metrics_list, labels=Y, verbose=False
        )
        all_mean_probs.append(mean_probs)
        for metric in metrics_list:
            all_metric_values[metric].append(metric_values[metric])
    
    probs = torch.cat(all_probs)
    Y = torch.cat(all_Y)
    Y_hat = probs.argmax(1)
    acc = (Y_hat == Y).float().mean().item()
    mean_probs = torch.cat(all_mean_probs).mean()
    metric_values = {metric: torch.cat(all_metric_values[metric]) for metric in metrics_list}

    # Courbes d'accuracy et calcul des pénalités
    penalties = {}
    accuracy_curves = {}
    for metric, color in zip([
        "variance_predicted", "variance_max",
        "predictive_entropy_predicted", "predictive_entropy_max"
    ], ["royalblue", "seagreen", "deeppink", "darkorchid"]):
        quantiles, thresholds, accuracies = accuracy_threshold(
            Y_hat, Y, metric_values[metric], metric_name=metric, color=color, display=False
        )
        accuracy_curves[metric] = {
            "quantiles": quantiles,
            "thresholds": thresholds,
            "accuracies": accuracies
        }
        penalty_iso = monotonicity_penalty(quantiles, accuracies, method="isotonic")
        penalty_rearr = monotonicity_penalty(quantiles, accuracies, method="rearrangement")
        penalties[metric] = {
            "penalty_isotone": penalty_iso,
            "penalty_rearrangement": penalty_rearr
        }

        if penalty_iso < min_penalties[metric]["penalty_isotone"]:
            min_penalties[metric]["penalty_isotone"] = penalty_iso
            min_penalties[metric]["config_ids_iso"] = [i]
        elif penalty_iso == min_penalties[metric]["penalty_isotone"]:
            min_penalties[metric]["config_ids_iso"].append(i)

        if penalty_rearr < min_penalties[metric]["penalty_rearrangement"]:
            min_penalties[metric]["penalty_rearrangement"] = penalty_rearr
            min_penalties[metric]["config_ids_rearr"] = [i]
        elif penalty_rearr == min_penalties[metric]["penalty_rearrangement"]:
            min_penalties[metric]["config_ids_rearr"].append(i)

    result = {
        "config_id": i,
        "dico_layers": dico_layers,
        "before": before,
        "acc": acc,
        "mc_mean_probs": mean_probs.item(), 
        **{metric: metric_values[metric].mean().item() if isinstance(metric_values[metric], torch.Tensor) else metric_values[metric] for metric in metrics_list},
        "penalties": penalties
    }
    all_results.append(result)
    all_accuracy_curves.append({
        "config_id": i,
        "accuracy_curves": accuracy_curves
    })

    if (i+1) % 10 == 0 or i == len(grid_configs)-1:
        print(f"Config {i+1}/{len(grid_configs)} traitée.")

# Sauvegarde au format pickle
with open("all_results.pkl", "wb") as f:
    pickle.dump(all_results, f)
print("Résultats sauvegardés dans all_results.pkl")

# Sauvegarde également en CSV
df_results = pd.DataFrame(all_results)
df_results.to_csv("all_results.csv", index=False)
print("Résultats sauvegardés dans all_results.csv")

for metric in metrics_list:
    print(f"Métrique '{metric}' :")
    print(f"  - Pénalité isotone minimisée pour config(s) {min_penalties[metric]['config_ids_iso']} avec valeur {min_penalties[metric]['penalty_isotone']}")
    print(f"  - Pénalité rearrangement minimisée pour config(s) {min_penalties[metric]['config_ids_rearr']} avec valeur {min_penalties[metric]['penalty_rearrangement']}")


In [ ]:
# rajouter affichage tableau avec double index comme pour l'autre notebook
# rajouter 3 colonnes pour la config : layer, proba, before pour faire une recherche (ex: before = true)
import ast

df_results = pd.read_csv("all_results.csv")

# Extraction des infos de config pour les colonnes d'index
def extract_layers(dico_layers_str):
    dico = ast.literal_eval(dico_layers_str)
    return ','.join(sorted(dico.keys()))

def extract_proba(dico_layers_str):
    dico = ast.literal_eval(dico_layers_str)
    # On suppose que toutes les couches d'une config ont la même proba
    return list(dico.values())[0] if dico else None

df_results['layers'] = df_results['dico_layers'].apply(extract_layers)
df_results['proba'] = df_results['dico_layers'].apply(extract_proba)
df_results['before'] = df_results['before'].astype(bool)

# Extraction des pénalités pour chaque métrique
metrics = [
    "variance_predicted",
    "variance_max",
    "predictive_entropy_predicted",
    "predictive_entropy_max"
]
penalties = ["penalty_isotone", "penalty_rearrangement"]

for metric in metrics:
    for penalty in penalties:
        df_results[(metric, penalty)] = df_results['penalties'].apply(
            lambda d: ast.literal_eval(d)[metric][penalty]
        )

# Création du MultiIndex pour les colonnes
columns = pd.MultiIndex.from_product([metrics, penalties])

# Création du MultiIndex pour les lignes
df_results.set_index(['layers', 'proba', 'before'], inplace=True)

# Trouver les valeurs minimales pour chaque métrique/penalty
min_vals = {}
for metric in metrics:
    for penalty in penalties:
        col = (metric, penalty)
        min_vals[col] = df_results[col].min()

# Fonction de surlignage
def highlight_min(s):
    is_min = s == min_vals[s.name]
    return ['background-color: #228B22' if v else '' for v in is_min]

# Affichage du tableau formaté avec surlignage
display(df_results[columns].sort_index().style.apply(highlight_min, subset=columns))